#### Student Name: Kumiko Komori
#### Student ID: A18547845

*AI Usage: Used Claude for all coding and conceptual portions of this assignment.*

## Speech Formants with Linear Predictive Coding, Vocoder (Mister Blue Sky)

Instructions: 

* This notebook is an interactive assignment; please read and follow the instructions in each cell. 

* Cells that require your input (in the form of code or written response) will have 'Question #' above.

* After completing the assignment, please submit this notebook and its pdf printout and all sound files. 

## Speech Formants and LPC

In this section, you will synthesize vowel sounds, and investigate the frequencies in vowels from your own voice. 

In [ ]:
import numpy as np
import pandas as pd
from collections import Counter
from numpy.random import multinomial as randm
from numpy import where
import scipy.signal as si
import matplotlib.pyplot as plt
from matplotlib import patches
import IPython.display as ipd
import librosa
import scipy
import librosa.display as ld
from scipy.io import wavfile as wavfile 
import copy


Fdict = {
    'mystery_1':[[328, 2208, 2885],[27,80,575]],
    'mystery_2':[[504, 868, 2654],[62,   108,  299]],
    'mystery_3':[[700, 1220, 2600],[130,   70,  160]]
    } # Formant frequencies in Hz

def excitation(f0, jitt, dur, nharm=None, unvoiced=False, sample_rate=None):
    sample_rate = fs if sample_rate is None else sample_rate
    nsamps = int(round(sample_rate * dur))
    assert nsamps > 0

    if unvoiced:
        sig = np.random.normal(size=nsamps)
    else:
        if nharm is None:
            nharm = int((sample_rate / 2) // f0)
        n = np.arange(nsamps)
        phase_jitter = np.random.uniform(size=nsamps) * 2 * np.pi
        omega0 = 2 * np.pi * f0 / sample_rate
        harmonics = np.arange(1, nharm)[:, None]
        sig = np.cos(
            harmonics * omega0 * n + jitt * phase_jitter
        ).sum(axis=0)

    peak = np.max(np.abs(sig))
    assert peak > 0
    return sig / peak

def voca(sig, F, Fb, sample_rate=None):
    sample_rate = fs if sample_rate is None else sample_rate
    F = np.asarray(F, dtype=float)
    Fb = np.asarray(Fb, dtype=float)

    radii = np.exp(-np.pi * Fb / sample_rate)
    angles = 2 * np.pi * F / sample_rate
    poles = radii * np.exp(1j * angles)
    B, A = si.zpk2tf(
        [],
        np.concatenate([poles, np.conj(poles)]),
        1.0,
    )

    speech = si.lfilter(B, A, np.asarray(sig, dtype=float))
    scale = np.std(speech)
    assert scale > 0
    return speech / scale, B, A

fs = 8192 # 22050  % Sampling rate in Hz ("telephone quality" for speed)

vowels = list(Fdict.keys())
f0 = 150 # Pitch in Hz
dur = 1 #one second in duration
ji = 0.1 #0.1
ex = excitation(f0,ji,dur)

text = ['mystery_1','mystery_2','mystery_3']

speech = np.zeros(1)
for t in text:
    F = np.array(Fdict[t][0])
    Fb = np.array(Fdict[t][1])
    print(t)

    vow,B,A = voca(ex,F,Fb)
    speech = np.concatenate((speech,vow))

speech = speech/np.std(speech)
plt.plot(speech)

In [ ]:
ipd.Audio(speech, rate=fs) 

##### Question 1 (10 points)

Based on the audio output, what vowels were synthesized as mystery_1, mystery_2, and mystery_3? 
Please specify using a word; for example, if you heard an 'oo' sound as in 'hoot', you may answer with the word "hoot". 

mystery_1:

mystery_2:

mystery_3:

Now we will examine just one vowel in greater detail. 
Select one mystery vowel to analyze below: 

In [ ]:
%matplotlib inline  

### Modify the line below:
text = ['mystery_1']

speech = np.zeros(1)
for t in text:
    F = np.array(Fdict[t][0])
    Fb = np.array(Fdict[t][1])
    print(t)

    vow,B,A = voca(ex,F,Fb)
    speech = np.concatenate((speech,vow))

speech = speech/np.std(speech)
plt.plot(speech)

In [5]:
# Plot the power spectral density (PSD)
plt.psd(speech, 1024)
plt.show()

In [ ]:
lpc_order = 10
s = speech

a = librosa.core.lpc(s, order=lpc_order)
print(a)
s_hat = scipy.signal.lfilter([0] + -1*a[1:], [1], s)
s_err = s[1:] - s_hat[:-1]
plt.plot(s[1:])
plt.plot(s_hat[:-1], linestyle='--')
plt.legend(['y', 'y_hat'])
plt.title('LP Model Forward Prediction')
plt.show()

In [ ]:
plt.plot(s[1:101])
plt.plot(s_hat[:100])
plt.legend(['y', 'y_hat'])
plt.show()

##### Question 2 (10 points)

What is being visualized as y and y_hat on the above plot?

``` Your response here ```

In [ ]:
w,h = si.freqz(b=1,a = a, fs=1)
plt.plot(w,20*np.log10(h))

In [ ]:
z,p,k = si.tf2zpk(B,A)
    
unit_circle = patches.Circle((0,0), radius=1, fill=False, color='black', ls='solid', alpha=0.9)
ax = plt.subplot(111)
ax.add_patch(unit_circle)
ax.spines['left'].set_position('center')
ax.spines['bottom'].set_position('center')
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
r = 1.5; plt.axis('scaled'); plt.axis([-r, r, -r, r])
ticks = [-1, -.5, .5, 1]; plt.xticks(ticks); plt.yticks(ticks)    
    
plt.plot(z.real, z.imag, 'ko', fillstyle='none', ms = 10)
plt.plot(p.real, p.imag, 'kx', fillstyle='none', ms = 10)

In [ ]:
D = np.abs(librosa.stft(s,n_fft=256,hop_length = 64))
ld.specshow(librosa.amplitude_to_db(D))

##### Question 3 (20 points)

Record yourself speaking the same vowel sound you analyzed above. 
Graph the power spectral density (PSD) of your recording alongside the PSD of the synthetic signal. 

In [ ]:
### Your Code Here
# Record yourself saying the same vowel (mystery_1) and save it next to this
# notebook, then update the filename below.
vowel_rec, vowel_sr = librosa.load("kmk_mystery_1.wav", sr=None, mono=True)

plt.figure()
plt.psd(vowel_rec, NFFT=1024, Fs=vowel_sr)
plt.psd(speech,    NFFT=1024, Fs=fs)
plt.legend(['recorded', 'synthetic'])
plt.title('PSD: recorded vs synthetic vowel (mystery_1)')
plt.xlabel('Frequency (Hz)')
plt.show()

##### Question 4 (10 points)

How does the power spectral density of your recorded signal compare to the LPC spectrum? 

``` Your response here ```

# Simple Singing Vocoder

In this section we will use a spoken sound to process an excitation that plays a melody. In music such an effect is known as vocoding and it is used to produce a talking musical instrument. 

In [ ]:
def lpc_to_formants(lpc, sr):
    roots = np.roots(lpc)
    roots = roots[np.imag(roots) > 0]
    roots = np.where(
        np.abs(roots) > 1,
        1 / np.conj(roots),
        roots,
    )

    freqs = np.angle(roots) * sr / (2 * np.pi)
    bws = -sr * np.log(np.abs(roots)) / np.pi

    valid = (
        np.isfinite(freqs)
        & np.isfinite(bws)
        & (freqs > 0)
        & (freqs < sr / 2)
        & (bws > 0)
    )
    order = np.argsort(freqs[valid])
    return freqs[valid][order], bws[valid][order]
    """Convert LPC to formants    
    """
        
    # extract roots, get angle and radius
    roots = np.roots(lpc)
    
    pos_roots = roots[np.imag(roots)>=0]
    if len(pos_roots)<len(roots)//2:
        pos_roots = list(pos_roots) + [0] * (len(roots)//2 - len(pos_roots))
    if len(pos_roots)>len(roots)//2:
        pos_roots = pos_roots[:len(roots)//2]
    
    w = np.angle(pos_roots)
    a = np.abs(pos_roots)
    
    order = np.argsort(w)
    w = w[order]
    a = a[order]
    
    freqs = w * (sr/(2*np.pi))
    bws =  -0.5 * (sr/(2*np.pi)) * np.log(a)    
    
    # exclude DC and sr/2 frequencies
    return freqs, bws

##### Questioon 5 [10 points] 

Record yourself speaking slowly the sentence "Mister Blue Sky". Plot a spectrogram of the speech sound.

In [ ]:
# Your code here
# Record yourself slowly saying "Mister Blue Sky" and save it (mono) next to this
# notebook, then update the filename below.
mbs, mbs_sr = librosa.load("kmk_mister_blue_sky.wav", sr=None, mono=True)

D = np.abs(librosa.stft(mbs, n_fft=512, hop_length=128))
plt.figure()
ld.specshow(librosa.amplitude_to_db(D, ref=np.max),
            sr=mbs_sr, hop_length=128, x_axis='time', y_axis='hz')
plt.colorbar(format='%+2.0f dB')
plt.title('"Mister Blue Sky" spectrogram')
plt.show()

##### Question 6 [30 points]

In this question we will create a song based on the spoken sentence you recorded. You will choose the melody by creating a sequence of pitches that change slowly over time.
Write a function that does the following:

1. Divide the speech signal into short slices (frames) of 512 samples with 50% overlap
2. For each speech segment compute formants by converting lpc_to_formants (F,Fb)
3. Choose a pitch (f0) for each segment
4. Using the voca function, create a speech sound (vow) from an excitation (ex) with that pitch
4. Overlap and add the sound segments with cross-fade window to create one long sound file 

You are free to alter the durations of the segments and choice of notes for the melody. 
Note that the notes should be relatively long (f0 should not change very often).

Cross-fade between segments can be done by applying a traingular (numpy.bartlett) or raised cosine (numpy.hanning) window to each segment before.

In [ ]:
# Read the required mono speech file.
# Use the same "Mister Blue Sky" recording you analyzed in Question 5 (mono).
sr, wave = wavfile.read("kmk_mister_blue_sky.wav")
assert wave.ndim == 1
wave = wave.astype(np.float64)

frame_len = 512
hop_length = frame_len // 2
num_frames = 1 + int(
    np.ceil(max(0, len(wave) - frame_len) / hop_length)
)
padded_len = (num_frames - 1) * hop_length + frame_len
padded_wave = np.pad(wave, (0, padded_len - len(wave)))

# Periodic Hann is COLA-compatible at 50% overlap.
window = scipy.signal.windows.hann(frame_len, sym=False)
vocode = np.zeros(padded_len, dtype=np.float64)

lpc_order = 10
# Melody (Hz); notes are held long so f0 changes slowly across the whole file.
melody = np.array([220.00, 246.94, 293.66, 329.63, 293.66, 246.94, 220.00])  # A3 B3 D4 E4 D4 B3 A3
frames_per_note = max(1, num_frames // len(melody))

for frame_index, start in enumerate(
    range(0, padded_len - frame_len + 1, hop_length)
):
    wave_slice = padded_wave[start:start + frame_len]

    ### Your Code Here
    # 1. LPC of this speech frame -> formants (F, Fb)
    if np.any(wave_slice):
        a = librosa.core.lpc(wave_slice, order=lpc_order)
        F, Fb = lpc_to_formants(a, sr)
    else:
        F, Fb = np.array([]), np.array([])   # silent (padding) frame

    # 2. Pick a pitch (f0) for this segment from the slowly-changing melody
    f0 = float(melody[min(frame_index // frames_per_note, len(melody) - 1)])

    # 3. Build an excitation exactly frame_len long (dur = frame_len / sr)
    ex = excitation(f0, ji, frame_len / sr, sample_rate=sr)
    assert len(ex) == frame_len

    vow, B, A = voca(ex, F, Fb, sample_rate=sr)
    assert len(vow) == frame_len
    vocode[start:start + frame_len] += vow * window

vocode = vocode[:len(wave)]
# Normalize to avoid clipping, then write the vocoded song.
vocode = vocode / (np.max(np.abs(vocode)) + 1e-12)
wavfile.write(
    "mister_blue_sky_vocoded.wav",
    sr,
    vocode.astype(np.float32),
)

##### Question 7 [10 points]

Why did we use overlapping windows for vocoder? 

``` your response here ```